In [1]:
#!/usr/bin/env python
"""
Обновлённый скрипт для валидации гипотез H1 и H2,
где в H2 для построения прогнозов и вычисления метрик:
  • точечный прогноз (mean) берётся из детерминированного forecast['yhat'],
    как раньше, чтобы MSE сравнивать с ним,
  • интервалы строятся через predictive_samples() (не влияют на MSE, но нужны
    для визуализации).

Результаты H1 и H2 (MSE/MAE) сохраняются в hypothesis_test_results.csv,
график сравнения MSE сохраняется в hypothesis_comparison.png.
Также строится файл hypotheses_overview.png (4 панели).
"""

import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------
# ЧАСТЬ 1: Валидация гипотез H1 и H2
# ---------------------------------------------------------------------

def evaluate_model(y_true, y_pred, model_name):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    print(f"\n{model_name} Performance:")
    print(f"MSE: {mse:.4f}")
    print(f"MAE: {mae:.4f}")
    return {'mse': mse, 'mae': mae}

def fit_pls_safely(X_tr, y_tr, X_te, n_components, return_model=False):
    """
    Fit PLS с уменьшением дочерей компонент, если не сходится.
    В fallback возвращает прогноз средним по y_tr.
    """
    try:
        eps = 1e-10
        X_tr_stable = X_tr + eps
        X_te_stable = X_te + eps

        pls = PLSRegression(n_components=n_components, scale=False)
        pls.fit(X_tr_stable, y_tr)
        if return_model:
            return pls.predict(X_te_stable), pls
        return pls.predict(X_te_stable)
    except:
        try:
            n_new = max(1, n_components - 1)
            print(f"Retrying PLS with {n_new} components...")
            pls = PLSRegression(n_components=n_new, scale=False)
            pls.fit(X_tr_stable, y_tr)
            if return_model:
                return pls.predict(X_te_stable), pls
            return pls.predict(X_te_stable)
        except:
            print("PLS fallback to mean prediction...")
            mean_pred = np.tile(np.mean(y_tr, axis=0), (X_te.shape[0], 1))
            if return_model:
                return mean_pred, None
            return mean_pred

print("Loading synthetic data for validation...")
df = pd.read_csv('synthetic_data_confirming.csv', parse_dates=['Time'])

# --- Подготовка матриц ---
vocab    = df['Topic'].unique()
clusters = df['Cluster'].unique()
dates    = df['Time'].unique()
num_dates = len(dates)
test_size = 2  # два периода прогноза (2 × 2W = 4 недели)

# Word frequency matrix: Time × Topic
word_matrix = pd.pivot_table(
    df, values='Count', index='Time', columns='Topic', aggfunc='sum'
).fillna(0)

# Cluster matrix: Time × Cluster
cluster_matrix = pd.pivot_table(
    df, values='Count', index='Time', columns='Cluster', aggfunc='sum'
).fillna(0)

# Train/test split
X = word_matrix.values                # (num_dates, num_words)
y = cluster_matrix.values             # (num_dates, num_clusters)
X_train = X[:-test_size]
y_train = y[:-test_size]
X_test  = X[-test_size:]
y_test  = y[-test_size:]

# Стандартизация признаков
scaler = StandardScaler(with_mean=False)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)


KeyboardInterrupt: 

In [ ]:

# =============================
# H1: Feature Selection через PLS
# =============================
print("\n=== Hypothesis 1: Feature Selection ===")
results = []

n_components = min(3, X_train_scaled.shape[1] - 1)
# 1) Full model (все признаки)
y_pred_full, pls_model = fit_pls_safely(
    X_train_scaled, y_train, X_test_scaled, n_components, return_model=True
)
full_metrics = evaluate_model(y_test, y_pred_full, "Full Model (All Features)")
results.append({"method": "Full Model", **full_metrics})

# 2) Importance по абс. коэффициентам PLS
coef_abs = np.abs(pls_model.coef_)
if coef_abs.shape[0] == X_train_scaled.shape[1]:
    importance = coef_abs.sum(axis=1)
else:
    importance = coef_abs.sum(axis=0)

top_k = int(len(vocab) * 0.2)   # 20% слов
selected_idx = np.argsort(importance)[::-1][:top_k]
selected_vocab = [vocab[i] for i in selected_idx]

# 3) PLS на отобранных признаках
X_train_sel_scaled = X_train_scaled[:, selected_idx]
X_test_sel_scaled  = X_test_scaled[:,  selected_idx]
X_train_sel_orig   = X_train[:, selected_idx]  # НЕ масштабированные

pls_selected = PLSRegression(
    n_components=min(3, X_train_sel_scaled.shape[1] - 1),
    scale=False
)
pls_selected.fit(X_train_sel_scaled, y_train)
y_pred_selected = pls_selected.predict(X_test_sel_scaled)
selected_metrics = evaluate_model(y_test, y_pred_selected, "Selected Features Model")
results.append({"method": "Selected Features", **selected_metrics})


In [ ]:

# =============================
# H2: Word-level Prediction → Cluster
# =============================
print("\n=== Hypothesis 2: Word-level vs Direct Prediction ===")

# 1) Direct Prophet: точечный прогноз (MAP) + predictive_samples (для интервала)
direct_means   = {}  # direct_means[cluster] = forecast['yhat'] на test_size
direct_samples = {}  # direct_samples[cluster] = array shape (n_sims, test_size)

for i, cluster in enumerate(clusters):
    # История кластера
    history_cluster = pd.DataFrame({
        'ds': dates[:-test_size],
        'y':  y_train[:, i]
    })
    model_dir = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        interval_width=0.95,
        changepoint_prior_scale=0.5,
        mcmc_samples=300
    )
    model_dir.fit(history_cluster)

    # Перспективный датасет
    future = model_dir.make_future_dataframe(periods=test_size, freq='2W')
    forecast_dir = model_dir.predict(future)

    # MAP-прогноз для тестовых точек
    direct_means[cluster] = forecast_dir.loc[
        forecast_dir['ds'].isin(dates[-test_size:]), 'yhat'
    ].values

    # Predictive samples (n_sims × (history+test_size))
    ps_dir = model_dir.predictive_samples(future)
    # Берём только последние test_size столбца
    direct_samples[cluster] = ps_dir['yhat'][:, -test_size:]

# 2) Word-based Prophet: для каждого выбранного слова берем MAP-прогноз
#    и predictive_samples, потом суммируем
word_means   = {}  # word_means[cluster] = sum of MAP-прогнозов по словам (test_size,)
word_samples = {}  # word_samples[cluster] = array shape (n_sims, test_size)

for i, cluster in enumerate(clusters):
    # Список индексов selected_vocab для текущего кластера
    idxs = [
        j for j, w in enumerate(selected_vocab)
        if df[df['Topic'] == w]['Cluster'].iloc[0] == cluster
    ]
    if not idxs:
        continue

    per_word_means = []  # список MAP-прогнозов (test_size,) для каждого слова
    per_word_samps = []  # список массивов (n_sims, test_size) для каждого слова

    for j in idxs:
        w = selected_vocab[j]
        history_word = pd.DataFrame({
            'ds': dates[:-test_size],
            'y':  X_train_sel_orig[:, j]
        })
        model_w = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=False,
            daily_seasonality=False,
            interval_width=0.95,
            changepoint_prior_scale=0.5,
            mcmc_samples=300
        )
        model_w.fit(history_word)

        future_w = model_w.make_future_dataframe(periods=test_size, freq='2W')
        forecast_w = model_w.predict(future_w)

        # 2a) MAP-прогноз для слова
        mean_w = forecast_w.loc[
            forecast_w['ds'].isin(dates[-test_size:]), 'yhat'
        ].values
        per_word_means.append(mean_w)

        # 2b) Predictive samples для слова
        ps_w = model_w.predictive_samples(future_w)
        per_word_samps.append(ps_w['yhat'][:, -test_size:])  # (n_sims, test_size)

    # 2a) Суммарный MAP-прогноз по словам (word-based mean)
    word_means[cluster] = np.sum(np.stack(per_word_means, axis=0), axis=0)

    # 2b) Суммируем семплы по каждому слову: shape (n_sims, test_size)
    stacked = np.stack(per_word_samps, axis=0)  # (num_words, n_sims, test_size)
    sum_samps = np.sum(stacked, axis=0)         # (n_sims, test_size)
    word_samples[cluster] = sum_samps

# 3) Считаем точечные прогнозы для метрик
cluster_predictions = np.zeros((test_size, len(clusters)))      # direct MAP
cluster_from_words  = np.zeros((test_size, len(clusters)))      # word-based MAP

for i, cluster in enumerate(clusters):
    # 3a) Direct MAP
    cluster_predictions[:, i] = direct_means[cluster]

    # 3b) Word-based MAP (sum of MAP каждого слова)
    if cluster in word_means:
        cluster_from_words[:, i] = word_means[cluster]
    else:
        # Если для кластера нет выбранных слов (крайний случай)
        cluster_from_words[:, i] = np.mean(y_train[:, i])

# 4) Вычисляем метрики MSE/MAE
direct_metrics = evaluate_model(y_test, cluster_predictions, "Direct Cluster Prediction")
results.append({"method": "Direct Cluster", **direct_metrics})

word_based_metrics = evaluate_model(
    y_test, cluster_from_words, "Word-based Prediction (Selected Words)"
)
results.append({"method": "Word-based", **word_based_metrics})

# Сохраняем результаты H1+H2
results_df = pd.DataFrame(results)
results_df.to_csv('hypothesis_test_results.csv', index=False)
print("\nResults saved to hypothesis_test_results.csv")

plt.figure(figsize=(12, 6))
plt.bar(results_df['method'], results_df['mse'], color=['gray', 'blue', 'orange', 'green'])
plt.title('MSE Comparison Across Methods')
plt.ylabel('Mean Squared Error')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('hypothesis_comparison.png')
plt.close()
print("Bar chart saved as hypothesis_comparison.png")



In [ ]:
# ---------------------------------------------------------------------
# ЧАСТЬ 2: Визуализация «hypotheses overview» (4 панели)
# ---------------------------------------------------------------------

OUT_PNG = "hypotheses_overview.png"
TEST_H  = test_size   # 2
TOP_SHOW= 15
CLUSTER = "Sports"

# 1) Фильтрация по CLUSTER
df_vis    = df[df["Cluster"] == CLUSTER].copy()
core      = [t for t in df_vis["Topic"].unique() if "_core" in t]
noise     = [t for t in df_vis["Topic"].unique() if "_noise" in t]
topics    = core + noise

# Матрица Time × Topic для выбранного кластера
mat       = df_vis.pivot_table(
    index="Time", columns="Topic", values="Count", aggfunc="sum"
).fillna(0)

cluster_series    = mat.sum(axis=1)       # суммарный ряд по всем словам
cluster_core_only = mat[core].sum(axis=1) # суммарный ряд только core

train_idx = slice(None, -TEST_H)
test_idx  = slice(-TEST_H, None)

fig = plt.figure(figsize=(11, 8))
gs  = fig.add_gridspec(2, 2)

# ---------------------------------------------------
# A: Heat-map корреляций (core vs noiseWN vs noiseSD)
# ---------------------------------------------------
ax = fig.add_subplot(gs[0, 0])

all_topics = mat.columns.tolist()
white_noise_topics = [t for t in all_topics if "_noiseWN" in t]
spike_decay_topics = [t for t in all_topics if "_noiseSD" in t]

n_show    = len(core)
sel_white = white_noise_topics[:n_show]
sel_spike = spike_decay_topics[:n_show]

sel_cols  = core + [CLUSTER] + sel_white + sel_spike

corr = mat.assign(**{CLUSTER: cluster_series}).corr()
c = ax.imshow(corr.loc[sel_cols, sel_cols],
              cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(sel_cols)))
ax.set_xticklabels(sel_cols, rotation=90, ha="right")
ax.set_yticks(range(len(sel_cols)))
ax.set_yticklabels(sel_cols)
ax.set_title("A. Pearson correlation (core vs noiseWN vs noiseSD)")
fig.colorbar(c, ax=ax, fraction=0.045)

# ---------------------------------------------------
# B: Суммарная траектория кластера (all vs core-only)
# ---------------------------------------------------
ax = fig.add_subplot(gs[0, 1])
ax.plot(cluster_series.index,      cluster_series.values,
        label="Cluster (ALL words)", lw=1.6)
ax.plot(cluster_core_only.index,   cluster_core_only.values,
        label="Cluster w/o noise",  ls="--", lw=1.6)
ax.set_title("B. Noise inflates cluster trajectory")
ax.legend()

# ---------------------------------------------------
# C: MAE Direct Prophet vs Sum of selected words
#    используем cluster_predictions и cluster_from_words
# ---------------------------------------------------
cluster_idx    = list(clusters).index(CLUSTER)
direct_pred_vis = cluster_predictions[:, cluster_idx]
sum_word_pred_vis = cluster_from_words[:, cluster_idx]

mae_direct_vis = mean_absolute_error(cluster_series.values[test_idx], direct_pred_vis)
mae_word_vis   = mean_absolute_error(cluster_series.values[test_idx], sum_word_pred_vis)

ax = fig.add_subplot(gs[1, 0])
ax.bar(
    ["Direct\n(cluster)", "Sum of\nselected words"],
    [mae_direct_vis, mae_word_vis],
    color=["tab:orange", "tab:green"]
)
ax.set_title("C. Prophet MAE on test horizon")
ax.set_ylabel("MAE")
ax.set_ylim(0, max(mae_direct_vis, mae_word_vis) * 1.25)
for ix, v in enumerate([mae_direct_vis, mae_word_vis]):
    ax.text(ix, v * 1.05, f"{v:.1f}", ha="center")

# ---------------------------------------------------
# D: |PLS coefficient| (top 15) для одного кластера
# ---------------------------------------------------
scaler_vis = StandardScaler(with_mean=False).fit(mat.values)
Xsc_vis    = scaler_vis.transform(mat.values)
pls_vis    = PLSRegression(n_components=3).fit(
    Xsc_vis[train_idx], cluster_series.values[train_idx, None]
)
coefs_vis  = pd.Series(
    np.abs(pls_vis.coef_).reshape(-1),
    index=mat.columns
)
top_imp_vis = coefs_vis.nlargest(TOP_SHOW)

ax = fig.add_subplot(gs[1, 1])
colors = ["tab:green" if t in core else "tab:gray" for t in top_imp_vis.index[::-1]]
ax.barh(top_imp_vis.index[::-1], top_imp_vis.values[::-1], color=colors)
ax.set_title("D. |PLS coefficient| (top 15)")
ax.set_xlabel("Importance")
ax.set_xlim(0, top_imp_vis.values.max() * 1.1)

plt.tight_layout()
plt.savefig(OUT_PNG, dpi=150)
print("✓ saved", OUT_PNG)
